In [1]:
import requests
from typing import List, Dict

def health_check(services: List[str], timeout: int = 3) -> Dict[str, str]:
    status = {}

    for service in services:
        try:
            response = requests.get(service, timeout=timeout)
            if response.status_code == 200:
                status[service] = "HEALTHY"
            else:
                status[service] = f"UNHEALTHY (Status: {response.status_code})"
        except requests.exceptions.RequestException as e:
            status[service] = f"DOWN ({str(e)})"

    return status


In [2]:
import requests
from typing import List, Dict

def health_check(services: List[str], timeout: int = 3) -> Dict[str, str]:
    status = {}

    for service in services:
        try:
            response = requests.get(service, timeout=timeout)
            if response.status_code == 200:
                status[service] = "HEALTHY"
            else:
                status[service] = f"UNHEALTHY (Status: {response.status_code})"
        except requests.exceptions.RequestException as e:
            status[service] = f"DOWN ({str(e)})"

    return status


In [3]:
services = [
    "https://google.com",
    "http://localhost:8080/health",
    "http://invalid-service"
]

results = health_check(services)

for service, result in results.items():
    print(f"{service} → {result}")


https://google.com → HEALTHY
http://localhost:8080/health → DOWN (HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002E2A253DE10>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it')))
http://invalid-service → DOWN (HTTPConnectionPool(host='invalid-service', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000002E2A2555990>: Failed to resolve 'invalid-service' ([Errno 11001] getaddrinfo failed)")))


In [11]:
import asyncio
import aiohttp
from typing import List, Dict

async def check_service(session: aiohttp.ClientSession, url: str, timeout: int) -> str:
    try:
        async with session.get(url, timeout=timeout) as response:
            if response.status == 200:
                return "HEALTHY"
            return f"UNHEALTHY (Status: {response.status})"
    except asyncio.TimeoutError:
        return "DOWN (Timeout)"
    except aiohttp.ClientError as e:
        return f"DOWN ({str(e)})"

async def async_health_check(services: List[str],timeout: int = 3) -> Dict[str, str]:
    results = {}
    async with aiohttp.ClientSession() as session:
        tasks = {
            service: asyncio.create_task(
                check_service(session, service, timeout)
            )
            for service in services
        }
        for service, task in tasks.items():
            results[service] = await task
    return results


In [12]:
if __name__ == "__main__":
    services = [
        "https://google.com",
        "http://localhost:8080/health",
        "http://invalid-service"
    ]
 #   results = asyncio.run(async_health_check(services))
    results = await async_health_check(services)
 
    for service, status in results.items():
        print(f"{service} → {status}")


https://google.com → HEALTHY
http://localhost:8080/health → DOWN (Cannot connect to host localhost:8080 ssl:default [Multiple exceptions: [Errno 10061] Connect call failed ('::1', 8080, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 8080)])
http://invalid-service → DOWN (Cannot connect to host invalid-service:80 ssl:default [getaddrinfo failed])
